In [ ]:
from app_parameters_config import (
    AMPLISEQ_TEST_ARGS,
    AMPLISEQ_CONDENSED_AMP_ARGS,
    AMPLISEQ_ITS_AMP_ARGS,
    AMPLISEQ_16S_AMP_ARGS 
)

## Enter TAPIS user and host machine information

To get things started, please run the following and enter the training account information provided to you:

In [ ]:
username = "andyyu"
password = "P#nguin777"

# UH Tenant
base_url = 'https://uhprod.uhtapis.org'

host_username = 'cmaiki_service'
host = 'koa.its.hawaii.edu'

## Authenticate and initialize Tapis v3 client

Using this information, you can now use `tapipy` to authenticate in the tenant and initialize the
Tapis v3 client. You should see your token information displayed. This may take a while to run but should take
no more than 30 seconds.

In [ ]:
from tapipy.tapis import Tapis
#Create python Tapis client for user
client = Tapis(base_url= base_url, username=username, password=password)
# *** Tapis v3: Call to Tokens API
client.get_tokens()
# Print Tapis v3 token
client.access_token

In [ ]:
# Export access_token to JWT env variable

import os
os.environ['JWT'] = client.access_token.access_token

## Systems

### Create a system for the HPC cluster

With just a few changes to the system definition you can create a system that can be used to run the
same application on an HPC type host. Note the minimal changes:

* **id** - A unique id is required
* **host** - Main hostname for the HPC system.
* **rootDir** - Using the root directory of the host gives us flexibility in setting **jobWorkingDir**.
  Note that you still need LINUX permissions.
* **jobWorkingDir** - Now determined dynamically using the Tapis v3 function HOST_EVAL()
* **jobRuntimes** - Most HPC systems support singularity and not docker
* **batchLogicalQueue.hpcQueueName** - HPC queue to use by default.
* **batchLogicalQueues** - HPC queue definitions for this HPC system.

### Schduler Profile
Typically not necessary

In [ ]:
# user_id = username
root_dir = "/mnt/lustre/koa/koastore/cmaiki_group"

system_id_hpc = "cmaiki-v2-koa-hpc"

# Create the system definition
exec_system_hpc = {
  "id": system_id_hpc,
  "description": "System for running C-MAIKI pipelines on Koa HPC cluster",
  "systemType": "LINUX",
  "host": host,
  "port": 2022,
  "defaultAuthnMethod": "PKI_KEYS",
  "effectiveUserId": "${apiUserId}",
  "rootDir": root_dir,
  "canExec": True,
  "jobRuntimes": [ { "runtimeType": "ZIP" } ],
  "jobWorkingDir": "${EffectiveUserId}",
  "canRunBatch": True,
  "batchScheduler": "SLURM",
  "batchDefaultLogicalQueue": "shared",
  "batchLogicalQueues": [
    {
      "name": "koa-shared",
      "hpcQueueName": "shared",
      "maxJobs": 50,
      "maxJobsPerUser": 10,
      "minNodeCount": 1,
      "maxNodeCount": 1,
      "minCoresPerNode": 1,
      "maxCoresPerNode": 1,
      "minMemoryMB": 3000,
      "maxMemoryMB": 32000,
      "minMinutes": 1,
      "maxMinutes": 4320
    }
  ]
}

# If you need to update the system, modify the above definition as needed
client.systems.patchSystem(**exec_system_hpc, systemId=system_id_hpc)

### Register Credentials for the HPC system

As before, now you will need to register credentials for your username. These will be used by Tapis to
access the host.

In [ ]:
private_key = os.getenv('MY_TAPIS_PRIVATE_KEY')
public_key = os.getenv('MY_TAPIS_PUBLIC_KEY')

In [ ]:
# Register credentials
client.systems.createUserCredential(systemId=system_id_hpc, userName=username, publicKey=public_key, privateKey=private_key)

In [ ]:
# client.files.mkdir(systemId=system_id_hpc, path="testdir")

In [ ]:
# client.systems.removeUserCredential(systemId=system_id_hpc, userName=username)

In [ ]:
# client.systems.getSystem(systemId=system_id_hpc)

In [ ]:
# client.access_token

In [ ]:
# client.files.listFiles(systemId=system_id_hpc, path="andyyu")

## Application

In order to run a job on a system you will need to create a Tapis application.

### Create an application that can be run on the VM host or the HPC cluster

### Demux App Def

In [ ]:
app_id_hpc = "demux-uhhpc"
output_dir = "${JobWorkingDir}/jobs/${JobUUID}"

app_def_hpc= {
    "id": app_id_hpc,
    "version": "1.0",
    "description": "Paired-end reads demultiplexer.",
    "runtime": "ZIP",
    "runtimeOptions": ["NONE"],
    "strictFileInputs": False,
    "containerImage": "/mnt/lustre/koa/lab/cmaiki_group/cmaiki_v2_apps/demux-app-v1.0.tar.gz",
    "jobType": "BATCH",
    "jobAttributes": {
        "execSystemId": system_id_hpc,
        "execSystemOutputDir": output_dir,
        "archiveOnAppError": True,
        "archiveSystemId": system_id_hpc,
        "archiveSystemDir": output_dir,
        "parameterSet": {
            "archiveFilter": {
                "includeLaunchFiles": True
            },
            "appArgs": [
#                 {"name": "outdir", "arg": f"--outdir demultiplexed", "description": f"Output path", "inputMode": "REQUIRED", "notes": {"Hidden": "true"}},
                {"name": "max_mismatches", "arg": "--max_mismatches 3", "description": f"Number of allowed mismatched with (combined-paired-end) barcode(s) sequence", "inputMode": "REQUIRED"},
                {"name": "n_per_file", "arg": "--n_per_file 100000", "description": f"Number of reads per file (demultiplexing is done in parallel on each sub-file)", "inputMode": "REQUIRED"},
                {"name": "n_bases", "arg": "--n_bases 100000", "description": "Number of bases to build the error model", "inputMode": "REQUIRED"},
                {"name": "matching", "arg": "--matching auto", "description": "Order to match index with barcodes. Choices: direct, reversed, auto", "inputMode": "REQUIRED"},
                {"name": "reverseComplement", "arg": "--reverseComplement", "description": "Reverse complement I2 (or I1 if reads are single-barcoded)", "inputMode": "INCLUDE_ON_DEMAND"},
                {"name": "singleBarcoded", "arg": "--singleBarcoded", "description": "Single barcoded reads", "inputMode": "INCLUDE_ON_DEMAND"}
            ]
        },
        "memoryMB": 16000,
        "nodeCount": 1,
        "coresPerNode": 1,
        "maxMinutes": 30
    },
}

In [ ]:
# client.apps.createAppVersion(**app_def_hpc)

client.apps.patchApp(**app_def_hpc, appId=app_id_hpc, appVersion='1.0')

In [ ]:
# client.apps.getApp(appId=app_id_hpc, appVersion='1.0')

### Check the status of the job


In [ ]:
# Check the status of the job

job_uuid_hpc = ""
print("****************************************************")
print(client.jobs.getJobStatus(jobUuid=job_uuid_hpc))
print("****************************************************")

In [ ]:
print(client.jobs.getJobHistory(jobUuid="a5f86709-1ffe-48d7-ae15-11c4f5c5f6fc-007"))